## Task 1: LangChain Setup & Core Concepts

In [ ]:
# Current compatible LangChain + Gemini stack.
# langchain-classic contains the requested AgentExecutor/create_tool_calling_agent APIs.
!pip -q install -U "langchain-classic==1.0.0" "langchain-google-genai==4.4.0" pandas pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

In [ ]:
import os
import io
import contextlib
import getpass
import pandas as pd

from langchain_core.tools import tool, ToolException
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

MODEL_NAME = "gemini-3.6-flash"

llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0,
)

print("LLM ready:", MODEL_NAME)

Enter your Gemini API key: ··········
LLM ready: gemini-3.6-flash


### Mapping LangChain concepts to Day 1's raw-Python equivalents

- Day 1 (raw Python + Gemini): client.models.generate_content(...)
  LangChain equivalent: ChatGoogleGenerativeAI (LLM wrapper)
  What it does: Wraps Gemini behind LangChain's Runnable interface

- Day 1 (raw Python + Gemini): Tool dict + Python function
  LangChain equivalent: @tool
  What it does: Builds the tool schema from the function signature and docstring

- Day 1 (raw Python + Gemini): Hand-written run_agent() loop
  LangChain equivalent: create_tool_calling_agent() + AgentExecutor
  What it does: Runs the reason -> act -> observe loop for us

- Day 1 (raw Python + Gemini): contents conversation list
  LangChain equivalent: RunnableWithMessageHistory
  What it does: Re-injects prior conversation turns automatically

### LCEL: a basic prompt-> response pipeline

In [ ]:
basic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise, helpful assistant."),
    ("human", "{question}"),
])

basic_chain = basic_prompt | llm | StrOutputParser()

print("Chain assembled:", basic_chain)

Chain assembled: first=ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a concise, helpful assistant.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]) middle=[ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.3.17', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini 3.6 Flash', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_

**What is the | pipe doing under the hood?** Each component implements LangChain's Runnable interface. The overloaded | operator builds a RunnableSequence, so the output of one component becomes the input to the next. The pipeline does not actually call Gemini until .invoke(), .stream(), or .batch() is used.

## Task 2 — Define & Register Tools

Three tools are used:

1. calculator: reused from Day 1.
2. get_weather: reused from Day 1's simulated weather tool.
3. get_product_price: new tool that reads a real local CSV file.

The tool docstrings are important because LangChain turns them into tool descriptions that Gemini can read when deciding whether a tool is appropriate.

In [ ]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression using numbers and +, -, *, /, and parentheses.
    Use this whenever the user asks for a calculation. Example: '25 * 8'."""
    import ast
    import operator as op

    allowed = {
        ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
        ast.USub: op.neg, ast.UAdd: op.pos,
    }

    def evaluate(node):
        if isinstance(node, ast.Expression):
            return evaluate(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.UnaryOp) and type(node.op) in allowed:
            return allowed[type(node.op)](evaluate(node.operand))
        if isinstance(node, ast.BinOp) and type(node.op) in allowed:
            return allowed[type(node.op)](evaluate(node.left), evaluate(node.right))
        raise ToolException(f"Unsupported expression: {expression}")

    try:
        return str(evaluate(ast.parse(expression, mode="eval")))
    except (SyntaxError, ValueError, TypeError) as e:
        raise ToolException(f"Could not evaluate '{expression}': {e}")

calculator.handle_tool_error = True
print(calculator.name, "->", calculator.description)

calculator -> Evaluate a basic arithmetic expression using numbers and +, -, *, /, and parentheses.
Use this whenever the user asks for a calculation. Example: '25 * 8'.


In [ ]:
_WEATHER_STUB = {
    "lahore": {"temperature_c": 34, "condition": "Sunny"},
    "karachi": {"temperature_c": 29, "condition": "Partly cloudy"},
    "islamabad": {"temperature_c": 27, "condition": "Cloudy"},
    "faisalabad": {"temperature_c": 33, "condition": "Sunny"},
}

@tool
def get_weather(city: str) -> str:
    """Return simulated (not live) weather for a city.
    Only Lahore, Karachi, Islamabad, and Faisalabad are in this stub dataset."""
    data = _WEATHER_STUB.get(city.strip().lower())
    if data is None:
        raise ToolException(f"No stub weather data for '{city}'.")
    return f"{data['temperature_c']}C, {data['condition']}"

get_weather.handle_tool_error = True
print(get_weather.name, "->", get_weather.description)

get_weather -> Return simulated (not live) weather for a city.
Only Lahore, Karachi, Islamabad, and Faisalabad are in this stub dataset.


In [ ]:
# New tool: reads from a real local data source.
PRODUCTS_CSV_PATH = "products.csv"

products_df = pd.DataFrame([
    {"product_name": "Wireless Mouse", "category": "Electronics", "price_usd": 19.99, "in_stock": True},
    {"product_name": "Mechanical Keyboard", "category": "Electronics", "price_usd": 79.99, "in_stock": True},
    {"product_name": "USB-C Hub", "category": "Electronics", "price_usd": 34.50, "in_stock": True},
    {"product_name": "Laptop Stand", "category": "Accessories", "price_usd": 29.99, "in_stock": True},
    {"product_name": "Noise Cancelling Headphones", "category": "Electronics", "price_usd": 149.99, "in_stock": False},
    {"product_name": "Webcam 1080p", "category": "Electronics", "price_usd": 45.00, "in_stock": True},
    {"product_name": "Desk Lamp", "category": "Home Office", "price_usd": 22.75, "in_stock": True},
    {"product_name": "Ergonomic Chair", "category": "Home Office", "price_usd": 249.00, "in_stock": True},
    {"product_name": "Monitor 27in", "category": "Electronics", "price_usd": 189.99, "in_stock": True},
    {"product_name": "Bluetooth Speaker", "category": "Electronics", "price_usd": 39.99, "in_stock": True},
])
products_df.to_csv(PRODUCTS_CSV_PATH, index=False)

@tool
def get_product_price(product_name: str) -> str:
    """Look up a product's price and stock status from the local product catalog CSV.
    Matching is case-insensitive and allows partial matches. Returns an error if not found."""
    catalog = pd.read_csv(PRODUCTS_CSV_PATH)
    matches = catalog[catalog["product_name"].str.contains(product_name, case=False, na=False, regex=False)]
    if matches.empty:
        raise ToolException(f"No product matching '{product_name}' found in the catalog.")
    row = matches.iloc[0]
    stock_status = "in stock" if bool(row["in_stock"]) else "out of stock"
    return f"{row['product_name']}: ${float(row['price_usd']):.2f} ({stock_status})"

get_product_price.handle_tool_error = True

tools = [calculator, get_weather, get_product_price]
print(f"Wrote {PRODUCTS_CSV_PATH} with {len(products_df)} products.")
print("Tools:", [t.name for t in tools])

Wrote products.csv with 10 products.
Tools: ['calculator', 'get_weather', 'get_product_price']


**How do tool docstrings function as part of the prompt?** The @tool decorator uses the function's type hints and docstring to build the tool schema. The docstring becomes the tool description that Gemini sees when choosing among tools. A vague description can therefore cause the same wrong-tool-selection problem that a vague input_schema["description"] caused in Day 1.

## Task 3: Build an Agent with create_tool_calling_agent / AgentExecutor

langchain-classic supplies the exact APIs requested by the task sheet. The Google integration is the current Gemini integration, so the old the older langchain-google-genai integration compatibility problem is avoided.

In [ ]:
agent_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant with access to tools. Use tools when they are useful. "
     "Do not invent tool results. Give the user a clear final answer."),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    return_intermediate_steps=True,
    max_iterations=6,
    handle_parsing_errors=True,
)

print("AgentExecutor ready.")

AgentExecutor ready.


In [ ]:
# Capture the full verbose trace for one multi-step run.
trace_buffer = io.StringIO()

with contextlib.redirect_stdout(trace_buffer):
    multi_step_result = agent_executor.invoke({
        "input": "What is 25 multiplied by 8, and what's the weather like in Lahore?"
    })

captured_trace = trace_buffer.getvalue()
print(captured_trace)
print("\nFinal answer:", multi_step_result["output"])

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(




> Entering new AgentExecutor chain...

Invoking: `calculator` with `{'expression': '25 * 8'}`


200
Invoking: `get_weather` with `{'city': 'Lahore'}`


34C, Sunny[{'type': 'text', 'text': '**25 multiplied by 8** is **200**.\n\nThe weather in **Lahore** is currently **34°C and Sunny**.', 'index': 0, 'extras': {'signature': 'Ev0BCvoBARFNMg8pDMbm2opvPJ+oN2/H2cToEB2ZVWC8vxIkBrOKYb5HVzPPshuCQZEn8DeJxPmeSzC9LaP2yduBYLh+kdnVvgIYhj8GkKD6Z/RlagHzdYnr9Rjr4J2e31dhoPuGsTW0ggZyxW+RPvpiQXkk1GP3zLx99otePVwot1UwhDoV1iN2Ol3Pahi124BNOG7NwidDmWz32srCG422p5lQ6kRppATuj9xUiEJNGwHF+eAusWdhNlscfVlH+Y4+s6YPVhNn5MINOGUmKVKTvJL3sJQxI9kPdXkKSyHVGiV/+XpjUHcqmCCCWLor4x/167tU6u6d9UPVUNejfg=='}}]

> Finished chain.


Final answer: [{'type': 'text', 'text': '**25 multiplied by 8** is **200**.\n\nThe weather in **Lahore** is currently **34°C and Sunny**.', 'index': 0, 'extras': {'signature': 'Ev0BCvoBARFNMg8pDMbm2opvPJ+oN2/H2cToEB2ZVWC8vxIkBrOKYb5HVzPPshuCQZEn8DeJxPmeSzC9LaP2yduBYLh+kdnVvgIYhj8GkKD6Z/RlagHzdYnr9Rjr4J

### Annotating the trace

- **Reason:** Gemini decides which action is needed inside the model call. The verbose LangChain trace does not expose private reasoning text.
- **Act:** Invoking: 'tool_name' with '{...}' shows the selected tool and arguments.
- **Observe:** Observation: ... shows the tool result returned to the agent.
- **Repeat/final:** AgentExecutor automatically repeats the loop until Gemini returns a final response or max_iterations is reached.

### Comparison to Day 1

The core **Reason -> Act -> Observe -> repeat** structure is the same as Day 1. The major difference is that Day 1 exposed the loop and conversation list directly, while LangChain hides much of the orchestration behind AgentExecutor; return_intermediate_steps=True gives us structured access to the important tool actions and observations without rebuilding the loop ourselves.

In [ ]:
# Enter API Key 2. This key will be used for Tasks 4–5
api_key_2 = getpass.getpass("Enter your Gemini API key 2: ")
os.environ["GOOGLE_API_KEY"] = api_key_2

# Rebuild the Gemini model and agent so they use API Key 2
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0,
)

agent = create_tool_calling_agent(llm, tools, agent_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    return_intermediate_steps=True,
    max_iterations=6,
    handle_parsing_errors=True,
)

print("Switched to API Key 2. Tasks 4–5 will use this key.")


Enter your Gemini API key 2: ··········
Switched to API Key 2. Tasks 4–5 will use this key.


## Task 4: Add Memory

RunnableWithMessageHistory stores conversation history by session ID and automatically injects that history into the agent prompt. This is the framework version of Day 1's manually maintained contents list.

In [ ]:
def _content_to_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                text = block.get("text")
                if text:
                    parts.append(str(text))
            elif isinstance(block, str):
                parts.append(block)
        return "".join(parts)
    return str(content)

def _normalize_agent_result(result):
    normalized = dict(result)
    normalized["output"] = _content_to_text(normalized.get("output", ""))
    return normalized

# Keeping AgentExecutor unchanged for Task 3. Only the memory path gets the normalization layer.
agent_executor_for_memory = agent_executor | RunnableLambda(_normalize_agent_result)

_session_store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in _session_store:
        _session_store[session_id] = InMemoryChatMessageHistory()
    return _session_store[session_id]

agent_with_memory = RunnableWithMessageHistory(
    agent_executor_for_memory,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="output",
)

print("Agent with memory ready.")


Agent with memory ready.


/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
# 3-turn conversation with follow-ups that depend on earlier context.
session_config = {"configurable": {"session_id": "budget-client-1"}}

turn_1 = agent_with_memory.invoke(
    {"input": "Find the price of the Wireless Mouse."},
    config=session_config,
)
print("Turn 1:", turn_1["output"])

turn_2 = agent_with_memory.invoke(
    {"input": "Now compare it to the Mechanical Keyboard."},
    config=session_config,
)
print("\nTurn 2:", turn_2["output"])

turn_3 = agent_with_memory.invoke(
    {"input": "Which one should I recommend to a budget-conscious client?"},
    config=session_config,
)
print("\nTurn 3:", turn_3["output"])



> Entering new AgentExecutor chain...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_product_price` with `{'product_name': 'Wireless Mouse'}`


Wireless Mouse: $19.99 (in stock)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The price of the Wireless Mouse is **$19.99** (currently in stock).', 'index': 0, 'extras': {'signature': 'Eu4BCusBARFNMg8V80AxvWqiJ7IgwJG1jKGDlxi9K4yKcloi5J3ugY77tgiYUapkeO+7hnMQvRo+OGVxn6GS18czN/XjS7FxQMDqz9ffUXvlz/+4K9KEPI4am7mAynewEyebC1WNnaLMwZFQoH7D1J/ILl1qnVkPD5+orE8BJfFEc/EWJnNspHVJMB0+pOL/6N7eSE8Y7xEjrM2F4xbLV+DNNUFwdSosbvYlF+54VLdQ3oSF4R5U44GWduyzzeT8NdijpLlKBF9KOGMMoufgZFpQZa1bomxPt8XRIUXzs3HBjASrTqPydQrQipdNdevPIQ=='}}]

> Finished chain.
Turn 1: The price of the Wireless Mouse is **$19.99** (currently in stock).


> Entering new AgentExecutor chain...


...


Invoking: `calculator` with `{'expression': '79.99 - 19.99'}`


60.0

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Here is the comparison between the two products:\n\n* **Wireless Mouse:** $19.99 (In stock)\n* **Mechanical Keyboard:** $79.99 (In stock)\n\n**Summary:**\n* The **Mechanical Keyboard** is **$60.00** more expensive than the Wireless Mouse.\n* The Mechanical Keyboard costs 4 times as much as the Wireless Mouse ($79.99 vs. $19.99).', 'index': 0, 'extras': {'signature': 'EqoBCqcBARFNMg+bMTQ34uPo4lcuZPqEp7sONiUUb/DuEQd0Ht/KPjzXZqlVp8ibvJsnqK+Ia1NTJDd5GXSP613c3067CFvXxlIIBV/Wf6LyzeCIskv5PMgz947ByIIOCH/ZK86cIGDSAlLkXtRJXCVz2W/WcI17Qa9dIQaGW9MA/CatrFoWwym+2MuEoE5HtsmRIaWoUHYvQ8K79z2zYDki63gFZXPNR4cpiL4='}}]

> Finished chain.

Turn 2: Here is the comparison between the two products:

* **Wireless Mouse:** $19.99 (In stock)
* **Mechanical Keyboard:** $79.99 (In stock)

**Summary:**
* The **Mechanical Keyboard** is **$60.00** more expensive than the Wireless Mouse.
* The Mechanical Keyboard costs 4 times as much as the Wireless Mouse ($79.99 vs. $19.99).


> Entering n

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'For a budget-conscious client, you should definitely recommend the **Wireless Mouse**. \n\nHere is why:\n* **Significant Savings:** At **$19.99**, it is **$60 cheaper** (75% less expensive) than the Mechanical Keyboard ($79.99).\n* **High Value for Essential Peripherals:** Standard wireless mice provide great daily productivity and convenience without a high premium.\n\n---\n\n### Additional Context to Keep in Mind:\n* **If they specifically need a keyboard:** Mechanical keyboards are generally a premium category. If the client needs a keyboard on a strict budget, a standard **membrane keyboard** (usually $15–$30) would be a much better budget fit than a $79.99 mechanical keyboard.\n* **If they need both:** Recommending a budget mouse ($19.99) alongside a standard non-mechanical keyboard will keep their total spend well under $50.', 'index': 0, 'extras': {'signature': 'EuIFCt8FARFNMg+WdHjWgTmOEEveGLsBKGXowiboZISC5VAH6xnwTM9d4r0kP3Hu0elirVP4Of2kWyEzA4WRIshblF7

If turn 3 correctly refers to both products without them being restated, the memory wiring is working. The important point is that the model receives the earlier conversation automatically; the user does not have to repeat the product names in every turn.

## Task 5: Structured Output & Error Handling

### Structured output

AgentExecutor produces a normal text answer. To demonstrate LangChain's structured-output support cleanly, we use a second small chain that converts the agent's final answer into a Pydantic model.

In [ ]:
class BudgetRecommendation(BaseModel):
    cheaper_item: str = Field(description="Name of the cheaper item")
    price_difference_usd: float = Field(description="Price difference between the two items in USD")
    recommendation: str = Field(description="One-sentence recommendation for a budget-conscious client")

structuring_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract a structured recommendation from the assistant's answer below."),
    ("human", "{agent_answer}"),
])

structuring_chain = structuring_prompt | llm.with_structured_output(BudgetRecommendation)

structured_result = structuring_chain.invoke({
    "agent_answer": turn_3["output"]
})

print(structured_result)
print(type(structured_result))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


cheaper_item='Wireless Mouse' price_difference_usd=60.0 recommendation='For a budget-conscious client, you should definitely recommend the Wireless Mouse.'
<class '__main__.BudgetRecommendation'>


### Error handling for tool failures

A tool can raise ToolException. Setting handle_tool_error converts that failure into a tool observation instead of allowing the exception to crash the agent. This is the framework equivalent of Day 1's try/except block that returned an error observation to Gemini.

In [ ]:
@tool
def flaky_price_lookup(product_name: str) -> str:
    """Look up a product price while simulating an intermittent upstream failure.
    Roughly one third of calls fail so error recovery can be observed."""
    import random
    if random.random() < (1 / 3):
        raise ToolException("Simulated upstream failure: catalog service timed out.")
    return get_product_price.func(product_name)

flaky_price_lookup.handle_tool_error = (
    "The catalog lookup failed. The agent should retry or ask about another product."
)

import random
random.seed(7)

for i in range(6):
    outcome = flaky_price_lookup.run({"product_name": "Wireless Mouse"})
    print(f"Call {i + 1}: {outcome}")

Call 1: The catalog lookup failed. The agent should retry or ask about another product.
Call 2: The catalog lookup failed. The agent should retry or ask about another product.
Call 3: Wireless Mouse: $19.99 (in stock)
Call 4: The catalog lookup failed. The agent should retry or ask about another product.
Call 5: Wireless Mouse: $19.99 (in stock)
Call 6: Wireless Mouse: $19.99 (in stock)


### What did LangChain make easier?

LangChain made tool-schema generation, the reason/act/observe loop, structured intermediate steps, and multi-session memory much easier than Day 1's hand-written orchestration. The main abstraction leak is that the framework still exposes provider-specific behavior underneath, especially around Gemini's tool-calling metadata and rapidly changing model integrations. Structured output also works most cleanly as a separate chain after the agent rather than as a single AgentExecutor switch. Compared with Day 1, there is less plumbing to maintain, but more framework behavior to understand when something goes wrong.